# 05 · Detección del cambio — LandTrendr

**Qué hace:** corre LandTrendr sobre la serie anual de `negTCW` para detectar, píxel a píxel,
**cuándo** cambia la trayectoria y **con qué magnitud**. Produce el mapa de degradación de pastizal.

**Qué NO hace:** no decide *qué tipo* de cambio fue cada quiebre (notebook 07), no lo valida
(notebook 09) y no juzga si hay que creerle (notebook 06). Aquí solo se detecta.

### Qué consume

| Entrada | De dónde |
|---|---|
| Área de estudio | Asset `cusco_01_aoi` (notebook 01) |
| Máscaras de pajonal y análisis | Asset `cusco_02_mascara` (notebook 02) |
| Serie anual de 7 índices + `n_obs` | Asset `cusco_04_serie_anual` (notebook 04) |

**Los tres son assets, no recálculos.** Este notebook no reconstruye la serie ni las máscaras: las
lee. Es deliberado y está explicado en la sección 3.

### Qué produce

| Salida | La consume |
|---|---|
| Asset `cusco_05_resultado` | **Notebook 06** (control) y **07** (tipo de cambio) |
| `outputs/05_deteccion_cambio/degradacion_segmentos.tif` | Trabajo local sin Earth Engine |
| `outputs/05_deteccion_cambio/negTCW_ajustado.tif` | Interpretación de trayectorias (notebook 09) |
| `outputs/05_deteccion_cambio/parametros_corrida.csv` | Reproducibilidad: sin esto los TIFF no se pueden interpretar |

---

## La asimetría de entrada y salida

Conviene tenerlo claro antes de leer el código, porque es lo que hace confuso trabajar con
LandTrendr.

**Entran 13 imágenes.** LandTrendr espera un `ee.ImageCollection` con una imagen por año. No acepta
una sola imagen multibanda: necesita la dimensión temporal explícita, con `system:time_start` en cada
elemento.

**Sale una sola imagen.** Pero cada píxel contiene una **matriz**, no un número. La banda
`LandTrendr` es un array 2D de 4 filas × 13 columnas:

| Fila | Contenido |
|---|---|
| 0 | Año |
| 1 | Valor original (el que le dimos) |
| 2 | Valor ajustado (la polilínea que el algoritmo cree que representa la trayectoria) |
| 3 | ¿Es vértice? (1 o 0) |

Más una banda `rmse` y, por cada índice **acompañante**, una banda `<nombre>_fit` con sus valores
ajustados en los mismos vértices.

Por eso hay un paso de **desempaquetado** que ocupa media libreta. La librería oficial (`lt-gee`)
resuelve esto en JavaScript; aquí va portada a Python.

## Dos reglas de la entrada que no se pueden romper

**1. El orden de las bandas decide todo.** LandTrendr usa **la primera banda de cada imagen para
segmentar**; las demás solo las ajusta a los vértices que encontró con la primera. Si el orden se
altera, el algoritmo segmenta con el índice equivocado y el resultado es basura silenciosa — no
falla, solo está mal. Por eso se hace `.select(INDICES)` explícito en vez de confiar en el orden que
traiga el asset.

**2. `n_obs` no entra.** La serie lleva `n_obs` como banda 8. Si va dentro de la colección,
LandTrendr la trata como índice acompañante e intenta ajustar un **conteo de observaciones** a la
trayectoria, lo que no significa nada. Va por separado, como máscara de calidad.

**Orden del proyecto:** 01 área → 02 cobertura → 03 fuego → 04 serie anual → **05 detección** →
06 control → 07 tipo de cambio → 08 clima → 09 validación → 10 entregables.

## 1 · Decisiones de este notebook

Seis decisiones que de otro modo quedarían enterradas en el código. Cada constante de la celda de
configuración corresponde a una de estas.

### 1.1 · Umbral mínimo de magnitud — derivado de los datos, no elegido

Es **la decisión que determina la cifra del entregable**. Separa degradación real de ruido: un
umbral laxo infla el resultado con variabilidad climática, uno estricto esconde la degradación leve,
que es justo lo que buscamos.

Elegirlo mirando cuál da el mapa más convincente sería indefendible. En vez de eso se **estima el
ruido de la serie**: se toman los píxeles que MapBiomas clasifica como pajonal en todos los años, se
calcula la desviación estándar de las diferencias año a año de `negTCW` en esa población, y el
umbral se fija en **2σ**.

La lógica es que un píxel estable solo debería variar por ruido —nubes residuales, fenología, lluvia
interanual— así que la dispersión de sus diferencias anuales *es* el piso de ruido del sensor y del
procesamiento juntos. Por debajo de 2σ la serie no distingue señal de ruido, y eso se puede escribir
tal cual en el informe: **"el cambio mínimo detectable con esta serie es X"**.

El supuesto que hay detrás, y que conviene declarar: se asume que la población estable de MapBiomas
es realmente estable. Si MapBiomas estuviera etiquetando como estable un pajonal que se degrada de
forma generalizada, el σ estaría inflado y el umbral saldría demasiado alto, haciéndonos
conservadores. Es un sesgo hacia subestimar, que es el lado seguro para un informe público.

### 1.2 · Duración mínima: 3 años

Un quiebre de un solo año en pastizal suele ser una quema o un año seco, no degradación. La
degradación por sobrepastoreo es un proceso sostenido. Se exige **duración ≥ 3 años** para el
indicador principal; los segmentos de 1–2 años se conservan aparte, porque son candidatos a quema y
eso lo resuelve el notebook 07 cruzando con el asset del notebook 03.

### 1.3 · Se extraen todos los segmentos, no solo uno

LandTrendr puede encontrar varios segmentos de pérdida en un mismo píxel. Guardarlos todos cuesta
poco y permite cambiar el criterio de "cuál es el principal" sin recalcular nada. El **principal** se
define como el de **mayor magnitud entre los que duran ≥ 3 años**.

### 1.4 · Se corre sobre toda la provincia, se enmascara al final

LandTrendr trabaja píxel a píxel, sin ninguna interacción espacial, así que enmascarar antes o
después da el mismo resultado en los píxeles que se conservan. Correr sobre los 530 km² completos
cuesta poco más y habilita un **control de cordura valioso**: si el algoritmo detecta "degradación"
con la misma frecuencia sobre zona urbana que sobre pajonal, algo está mal en el método. Ese control
se pierde si se enmascara antes, y es lo primero que hace el notebook 06.

### 1.5 · `maxSegments` = 4, con sensibilidad verificada en el notebook 06

El default de LT-GEE es 6, calibrado para bosque y series de 25–35 años. Con 13 observaciones, 6
segmentos permiten 7 vértices y el algoritmo termina siguiendo el ruido climático interanual. Se fija
en 4 y el notebook 06 compara 3, 4 y 5 para verificar que el valor no esté sobresegmentando.

### 1.6 · Los píxeles de mal ajuste se marcan, no se eliminan

El RMSE sale como banda de LandTrendr. Un RMSE alto significa que la polilínea no representa bien la
serie, así que el quiebre detectado ahí es poco confiable. Pero se guarda como **banda de confianza**
por percentil, no como máscara dura: así la decisión de dónde cortar queda en el notebook 10,
revisable, en vez de enterrada aquí.

**Y una advertencia sobre el RMSE:** no es una medida de validación. Solo dice cuánto se pega la
polilínea a la serie de entrada, y un modelo sobresegmentado baja el RMSE mientras empeora la
detección real. Sirve de diagnóstico por píxel, nada más.

### Lo que queda pendiente: la calibración de parámetros

Los valores de LandTrendr que usa este notebook vienen de la literatura, no de calibración contra
esta zona. La calibración correcta exige **puntos interpretados a mano** —150 a 300 píxeles
etiquetados con año y tipo de cambio— eligiendo por F1 de detección de vértices, no por RMSE.

Esos puntos son los del notebook 09, que viene después: hay un bucle de dependencia. Se cierra
interpretando un conjunto pequeño de **calibración**, separado espacialmente de los de validación,
antes de fijar los parámetros definitivos. Mientras tanto este notebook corre con los valores de
partida y el informe debe decirlo así.

## 2 · Configuración

In [ ]:
from datetime import date
from pathlib import Path

import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GEE_PROJECT = "viirs-peru"

# --- Assets de los notebooks anteriores ------------------------------------
ASSET_AOI = f"projects/{GEE_PROJECT}/assets/cusco_01_aoi"
ASSET_MASCARA = f"projects/{GEE_PROJECT}/assets/cusco_02_mascara"
ASSET_SERIE = f"projects/{GEE_PROJECT}/assets/cusco_04_serie_anual"
ASSET_RESULTADO = f"projects/{GEE_PROJECT}/assets/cusco_05_resultado"

ANIO_INI, ANIO_FIN = 2013, 2025
ANIOS = list(range(ANIO_INI, ANIO_FIN + 1))

INDICE_PRIMARIO = "negTCW"
INDICES_FTV = ["NDVI", "MSAVI", "TCG", "TCB", "BSI", "NDMI"]
INDICES = [INDICE_PRIMARIO] + INDICES_FTV     # el ORDEN importa: banda 0 = segmentación

# --- Parámetros de LandTrendr (decisión 1.5 · a calibrar) ------------------
PARAMS = {
    "maxSegments": 4,                 # default LT-GEE = 6, calibrado para bosque y 25-35 años
    "spikeThreshold": 0.9,
    "vertexCountOvershoot": 3,
    "preventOneYearRecovery": True,   # en pastizal un rebote de 1 año es lluvia, no recuperación
    "recoveryThreshold": 0.25,
    "pvalThreshold": 0.05,
    "bestModelProportion": 0.75,
    "minObservationsNeeded": 6,
}

# --- Umbrales del mapa de cambio (decisiones 1.1 y 1.2) --------------------
UMBRAL_MAG = None     # None = se deriva de los datos en la sección 4. Un número lo fija a mano.
SIGMAS = 2            # el umbral es SIGMAS * sigma del ruido en píxeles estables
DUR_MIN = 3           # años. Duración mínima para el indicador principal de degradación
DUR_QUEMA_MAX = 2     # segmentos de <= esta duración se guardan aparte (candidatos a quema)

OBS_MIN_ANIO = 3      # observaciones limpias por año para considerar el año bueno

ESCALA = 30
GRID_CRS = "EPSG:32719"
# 'Computed image is too large' aparece cuando el cómputo por tesela excede la memoria.
# tileScale lo parte en teselas más chicas: más lento, pero entra. Si vuelve a fallar, 8 o 16.
TILE_SCALE = 4
# Las estadísticas de superficie no necesitan 30 m: sobre 530 km², 60 m da el mismo km²
# con la cuarta parte de los píxeles. Los mapas y la exportación sí van a 30 m.
ESCALA_STATS = 60


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def con_reintentos(fn, intentos=6, espera=5):
    """Reintenta ante errores transitorios de Earth Engine."""
    import time
    for i in range(intentos):
        try:
            return fn()
        except Exception as e:
            msg = repr(e)
            if not any(t in msg for t in ["concurrent", "Too many", "Timed out", "timed out",
                                          "quota", "503", "500", "Connection"]) or i == intentos - 1:
                raise
            pausa = espera * (i + 1)
            print(f"\n    reintento {i + 1} en {pausa}s · {msg[:90]}")
            time.sleep(pausa)


def evaluar_serie(objetos, etiquetas=None, pausa=0.4):
    """getInfo() uno por uno: varias reducciones a la vez disparan
    'Too many concurrent aggregations'."""
    import time
    etiquetas = etiquetas or [str(i) for i in range(len(objetos))]
    salida = []
    for i, (obj, et) in enumerate(zip(objetos, etiquetas), 1):
        print(f"  [{i}/{len(objetos)}] {et}", end="  ", flush=True)
        salida.append(con_reintentos(lambda o=obj: o.getInfo()))
        print("ok", flush=True)
        if pausa and i < len(objetos):
            time.sleep(pausa)
    return salida


iniciar_gee()
RAIZ = buscar_raiz()
SALIDA = RAIZ / "outputs" / "05_deteccion_cambio"

print("Raíz del proyecto :", RAIZ)
print("Periodo           :", ANIOS[0], "-", ANIOS[-1], f"({len(ANIOS)} años)")
print("Índice primario   :", INDICE_PRIMARIO, "(banda 0, la que segmenta)")
print("Acompañantes      :", ", ".join(INDICES_FTV))
print("Parámetros        :", PARAMS)
CONFIG_LISTA = True

## 3 · Cargar los insumos desde los assets

Las tres entradas se **leen**, no se recalculan. Las razones, que son distintas en cada caso:

**La serie anual.** Earth Engine es perezoso y no guarda resultados intermedios. Si la serie se
reconstruyera aquí, cada número que pidiera este notebook volvería a ejecutar la receta completa
desde las escenas crudas —filtrar, enmascarar, calcular distancias multibanda, elegir el medoide,
siete índices, por trece años— y encima correría LandTrendr y manipularía arrays. El cómputo por
tesela desborda la memoria y aparece `Computed image is too large`. No es un problema de tamaño del
área: son 530 km². Es profundidad de la cadena.

**Las máscaras.** Una versión anterior las reconstruía aquí con su propia copia de `mascara_clase()`.
Eran dos definiciones separadas de lo mismo: si se cambiaba `MASCARA_MODO` en el notebook 02 y no
aquí, los dos notebooks medían sobre poblaciones distintas **sin avisar de nada**. Leer el asset
elimina esa posibilidad por construcción.

**El área de estudio.** Mismo argumento: una sola definición del polígono, producida por el
notebook 01.

Si algún asset no existe todavía, la celda lo dice y nombra el notebook que hay que ejecutar. No
reconstruye nada por su cuenta: hacerlo reintroduciría el problema que acabamos de describir.

In [ ]:
faltan = []

# --- Área de estudio -------------------------------------------------------
try:
    aoi = ee.FeatureCollection(ASSET_AOI).geometry()
    area_km2 = con_reintentos(lambda: aoi.area(maxError=10).divide(1e6).getInfo())
    print(f"✓ AOI          · {area_km2:,.1f} km²")
except Exception as e:
    faltan.append(("cusco_01_aoi", "notebook 01, sección 8", repr(e)[:90]))
    aoi = None

# --- Máscaras --------------------------------------------------------------
try:
    mascaras = ee.Image(ASSET_MASCARA)
    pastizal = mascaras.select("pastizal")
    MASCARA_ANALISIS = mascaras.select("analisis")
    meta = con_reintentos(lambda: mascaras.toDictionary().getInfo())
    print(f"✓ Máscaras     · modo '{meta.get('mascara_modo')}', "
          f"clase {meta.get('clase_objetivo')}, excluidas: {meta.get('clases_excluidas')}")
except Exception as e:
    faltan.append(("cusco_02_mascara", "notebook 02, sección 10", repr(e)[:90]))
    pastizal = MASCARA_ANALISIS = None

# --- Serie anual -----------------------------------------------------------
try:
    serie = ee.ImageCollection(ASSET_SERIE)
    n_img = con_reintentos(lambda: serie.size().getInfo())
    if n_img == 0:
        raise RuntimeError("la colección existe pero está vacía")
    # 'year' puede no sobrevivir la exportación: se re-deriva de system:time_start
    serie = serie.map(lambda img: img.set(
        "year", ee.Date(ee.Number(img.get("system:time_start"))).get("year")))
    print(f"✓ Serie anual  · {n_img} imágenes")
except Exception as e:
    faltan.append(("cusco_04_serie_anual", "notebook 04, sección 9", repr(e)[:90]))
    serie = None

if faltan:
    print("\n" + "=" * 72)
    print("FALTAN ASSETS. Este notebook no los reconstruye a propósito (ver sección 3).")
    print("=" * 72)
    for nombre, donde, err in faltan:
        print(f"  {nombre}")
        print(f"     generarlo en: {donde}")
        print(f"     error: {err}")
    raise RuntimeError("Ejecuta las celdas de exportación indicadas y vuelve a empezar.")

In [ ]:
# Verificación de la estructura que espera LandTrendr
primera = ee.Image(serie.sort("system:time_start").first())
bandas_serie = con_reintentos(lambda: primera.bandNames().getInfo())

print("Estructura de la serie:")
print("  imágenes          :", n_img, f"(se esperan {len(ANIOS)})")
print("  bandas por imagen :", bandas_serie)
print("  banda 0           :", bandas_serie[0],
      "✓" if bandas_serie[0] == INDICE_PRIMARIO else f"✗ debería ser {INDICE_PRIMARIO}")
print("  system:time_start :",
      "✓" if con_reintentos(lambda: primera.get("system:time_start").getInfo()) else "✗ falta")

# Calidad por píxel: cuántos años tienen suficientes observaciones limpias
n_obs_buenos = ee.ImageCollection([
    ee.Image(serie.filter(ee.Filter.eq("year", a)).first())
    .select("n_obs").gte(OBS_MIN_ANIO).rename("ok") for a in ANIOS
]).sum().rename("anios_buenos")

CALIDAD = n_obs_buenos.gte(PARAMS["minObservationsNeeded"])
MASCARA_FINAL = MASCARA_ANALISIS.And(CALIDAD).selfMask()


def km2_de(mascara, escala=None):
    """Superficie en km². tileScale evita 'Computed image is too large'."""
    v = (ee.Image.pixelArea().divide(1e6).updateMask(mascara)
         .reduceRegion(ee.Reducer.sum(), aoi, escala or ESCALA_STATS,
                       maxPixels=1e10, bestEffort=True, tileScale=TILE_SCALE))
    return ee.Number(v.get("area"))


print("\nSuperficie de cada máscara:")
sup = evaluar_serie(
    [km2_de(pastizal.selfMask()), km2_de(MASCARA_ANALISIS.selfMask()), km2_de(MASCARA_FINAL)],
    ["pajonal estable", "máscara de análisis", "análisis + calidad"])

for et, v in zip(["pajonal estable (clase 12 todos los años)",
                  "máscara de análisis (sin exclusiones ni bordes)",
                  f"análisis + ≥{PARAMS['minObservationsNeeded']} años buenos"], sup):
    print(f"  {et:50s} {v:8.2f} km²  ({int(v * 1e6 / ESCALA**2):,} px)")

km_pajonal = float(sup[2])

## 4 · El umbral de magnitud, derivado del ruido de la serie

La decisión 1.1 explica el razonamiento. Aquí se calcula.

**Procedimiento:** sobre los píxeles de pajonal estable según MapBiomas, se toman las 12 diferencias
año a año de `negTCW`, se calcula su desviación estándar por píxel, y se resume esa distribución
sobre la máscara. El umbral queda en `SIGMAS × σ`.

**Lo que mide:** cuánto se mueve un píxel que (según MapBiomas) no cambió de clase. Eso es ruido
—nubes residuales, fenología, lluvia interanual, error de registro— y marca el piso por debajo del
cual esta serie no puede distinguir señal.

Se reporta la mediana de σ y no la media porque la distribución tiene cola: unos pocos píxeles muy
ruidosos (bordes, nubes persistentes) arrastrarían la media hacia arriba y el umbral saldría
demasiado alto.

In [ ]:
def serie_banda(banda):
    """Lista ordenada de imágenes de una banda, un elemento por año."""
    return [ee.Image(serie.filter(ee.Filter.eq("year", a)).first()).select(banda).rename("v")
            for a in ANIOS]


capas = serie_banda(INDICE_PRIMARIO)
difs = ee.ImageCollection([capas[i + 1].subtract(capas[i]).rename("d")
                           for i in range(len(capas) - 1)])

sigma_px = difs.reduce(ee.Reducer.stdDev()).rename("sigma").updateMask(pastizal.selfMask())

est = con_reintentos(lambda: sigma_px.reduceRegion(
    reducer=(ee.Reducer.median()
             .combine(ee.Reducer.mean(), sharedInputs=True)
             .combine(ee.Reducer.percentile([10, 25, 75, 90]), sharedInputs=True)),
    geometry=aoi, scale=ESCALA_STATS, maxPixels=1e10, bestEffort=True,
    tileScale=TILE_SCALE).getInfo())

sigma_mediana = est["sigma_median"]
umbral_derivado = SIGMAS * sigma_mediana

print(f"RUIDO DE {INDICE_PRIMARIO} EN PAJONAL ESTABLE (σ de las diferencias año a año)")
print("=" * 66)
for k, etiqueta in [("sigma_p10", "percentil 10"), ("sigma_p25", "percentil 25"),
                    ("sigma_median", "MEDIANA"), ("sigma_p75", "percentil 75"),
                    ("sigma_p90", "percentil 90"), ("sigma_mean", "media")]:
    marca = "  <<<" if k == "sigma_median" else ""
    print(f"  {etiqueta:14s} σ = {est[k]:.4f}{marca}")
print("=" * 66)
print(f"\nUmbral = {SIGMAS} × σ mediana = {umbral_derivado:.4f}")

if UMBRAL_MAG is None:
    UMBRAL_MAG = umbral_derivado
    print(f"UMBRAL_MAG se fija en {UMBRAL_MAG:.4f} (derivado de los datos)")
else:
    print(f"UMBRAL_MAG estaba fijado a mano en {UMBRAL_MAG:.4f} "
          f"(el derivado habría sido {umbral_derivado:.4f})")

print(f"\nLectura para el informe: el cambio mínimo detectable de esta serie es")
print(f"{UMBRAL_MAG:.4f} unidades de {INDICE_PRIMARIO}. Por debajo de eso no se puede")
print("separar degradación de ruido, así que no se reporta.")

hist = con_reintentos(lambda: sigma_px.reduceRegion(
    ee.Reducer.histogram(maxBuckets=60), aoi, ESCALA_STATS,
    maxPixels=1e10, bestEffort=True, tileScale=TILE_SCALE).getInfo())["sigma"]

plt.figure(figsize=(9, 3.2))
plt.bar(hist["bucketMeans"], hist["histogram"],
        width=hist["bucketWidth"] * 0.9, color="#1a6fc0")
plt.axvline(sigma_mediana, color="#c62828", ls="--", label=f"σ mediana = {sigma_mediana:.4f}")
plt.axvline(umbral_derivado, color="#2e7d32", ls="-", lw=2,
            label=f"umbral = {SIGMAS}σ = {umbral_derivado:.4f}")
plt.xlabel(f"σ de las diferencias anuales de {INDICE_PRIMARIO}")
plt.ylabel("píxeles")
plt.title("Ruido de la serie en pajonal estable")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 · Correr LandTrendr

La colección se pasa con `.select(INDICES)` explícito para garantizar que la banda 0 sea `negTCW` y
que `n_obs` quede fuera (las dos reglas del encabezado).

La celda imprime los nombres de banda de la salida en vez de asumirlos. Y hay un detalle que importa:
**el índice primario no tiene banda `_fit`.** Solo los acompañantes la tienen. Los valores ajustados
del índice que segmenta están en la fila 2 del array `LandTrendr`, porque son el resultado de la
segmentación misma y no un ajuste derivado de ella. Las funciones `serie_ajustada()` y
`serie_original()` los extraen de ahí.

In [ ]:
def correr_landtrendr(coleccion, params=None, anios=None):
    """Corre LandTrendr garantizando el orden de bandas (banda 0 = índice de segmentación)."""
    p = {**PARAMS, **(params or {})}
    col = coleccion.select(INDICES)        # banda 0 = negTCW, sin n_obs
    if anios:
        col = col.filter(ee.Filter.inList("year", anios))
    return ee.Algorithms.TemporalSegmentation.LandTrendr(
        timeSeries=col.sort("system:time_start"), **p)


lt = correr_landtrendr(serie)

bandas_lt = con_reintentos(lambda: lt.bandNames().getInfo())
print("Bandas de salida de LandTrendr:")
for b in bandas_lt:
    nota = {"LandTrendr": "array 2D: [años, originales, ajustados, ¿vértice?]",
            "rmse": "error del ajuste, diagnóstico por píxel"}.get(b, "serie ajustada (FTV)")
    print(f"  {b:16s} · {nota}")

ftv_disponibles = [b for b in bandas_lt if b not in ("LandTrendr", "rmse")]
print(f"\nBandas ajustadas disponibles ({len(ftv_disponibles)}):", ftv_disponibles)
print(f"'{INDICE_PRIMARIO}' NO está en esa lista, y es lo esperado: sus valores ajustados")
print("salen de la fila 2 del array (funciones de abajo).")


def serie_ajustada(lt_img, anios=None):
    """Serie ajustada del índice PRIMARIO: fila 2 del array, una banda por año."""
    anios = anios or ANIOS
    return (lt_img.select("LandTrendr").arraySlice(0, 2, 3)
            .arrayProject([1]).arrayFlatten([[str(a) for a in anios]]))


def serie_original(lt_img, anios=None):
    """Serie original tal como la recibió LandTrendr: fila 1 del array."""
    anios = anios or ANIOS
    return (lt_img.select("LandTrendr").arraySlice(0, 1, 2)
            .arrayProject([1]).arrayFlatten([[f"o{a}" for a in anios]]))


fit = serie_ajustada(lt)
print(f"\nserie_ajustada(lt) → {len(ANIOS)} bandas, una por año")

## 6 · Desempaquetar los arrays

Esta es la parte que la librería oficial resuelve en JavaScript y aquí va portada. La lógica, paso a
paso:

1. **Quedarse con los vértices.** La fila 3 del array dice si cada año es vértice. `arrayMask` deja
   solo esas columnas, así que de 13 años se pasa a *n* vértices.
2. **Emparejar vértices consecutivos.** Un segmento va de un vértice al siguiente. Se toma el array
   de vértices sin el último (`izq`) y sin el primero (`der`): columna a columna, cada par
   `izq[i]`–`der[i]` es un segmento.
3. **Calcular los atributos** de cada segmento: año y valor de inicio y fin, duración, magnitud
   (`valor_fin − valor_inicio`) y tasa (`magnitud / duración`).
4. **Apilar todo** en un array de 7 filas × *n* segmentos.

Como `negTCW` está invertido, **magnitud positiva = pérdida de vegetación**. Si se hubiera usado TCW
sin invertir, todos los signos estarían al revés y el mapa reportaría recuperaciones donde hay
degradación. Es el error más común con esta herramienta.

La función `atributo()` convierte una fila del array en una banda normal con
`arrayProject([1]).arrayFlatten(...)`, el mismo truco de `lt-gee`. En los píxeles sin ningún segmento
que cumpla los filtros, el array queda vacío y el resultado sale enmascarado — que es exactamente lo
que se quiere.

In [ ]:
FILAS_SEG = ["anio_ini", "anio_fin", "val_ini", "val_fin", "magnitud", "duracion", "tasa"]


def datos_segmentos(lt_img):
    """Todos los segmentos de la salida de LandTrendr, como array de 7 filas × n segmentos.

    Port de getSegmentData() de lt-gee (JavaScript).
    """
    arr = lt_img.select("LandTrendr")
    es_vertice = arr.arraySlice(0, 3, 4)       # fila 3: ¿es vértice?
    vertices = arr.arrayMask(es_vertice)       # solo las columnas que son vértice

    izq = vertices.arraySlice(1, 0, -1)        # todos menos el último
    der = vertices.arraySlice(1, 1)            # todos menos el primero

    anio_ini, anio_fin = izq.arraySlice(0, 0, 1), der.arraySlice(0, 0, 1)
    val_ini, val_fin = izq.arraySlice(0, 2, 3), der.arraySlice(0, 2, 3)   # fila 2 = ajustado
    duracion = anio_fin.subtract(anio_ini)
    magnitud = val_fin.subtract(val_ini)       # negTCW invertido -> positivo = pérdida
    tasa = magnitud.divide(duracion)

    return ee.Image.cat([anio_ini, anio_fin, val_ini, val_fin,
                         magnitud, duracion, tasa]).toArray(0)


def atributo(arr, nombre):
    """Convierte una fila del array de segmentos en una banda normal."""
    i = FILAS_SEG.index(nombre)
    return arr.arraySlice(0, i, i + 1).arrayProject([1]).arrayFlatten([[nombre]])


def filtrar_perdidas(arr, umbral, dur_min=None, dur_max=None):
    """Deja solo los segmentos de pérdida que cumplen magnitud y duración."""
    mag = arr.arraySlice(0, FILAS_SEG.index("magnitud"), FILAS_SEG.index("magnitud") + 1)
    dur = arr.arraySlice(0, FILAS_SEG.index("duracion"), FILAS_SEG.index("duracion") + 1)
    cumple = mag.gt(umbral)
    if dur_min is not None:
        cumple = cumple.And(dur.gte(dur_min))
    if dur_max is not None:
        cumple = cumple.And(dur.lte(dur_max))
    return arr.arrayMask(cumple)


def principal(arr):
    """El segmento de mayor magnitud: ordena descendente y toma la primera columna."""
    clave = arr.arraySlice(0, FILAS_SEG.index("magnitud"),
                           FILAS_SEG.index("magnitud") + 1).toArray(0).multiply(-1)
    return arr.arraySort(clave).arraySlice(1, 0, 1)


segmentos = datos_segmentos(lt)
print("Funciones de desempaquetado listas.")
print("Filas del array de segmentos:", FILAS_SEG)
print(f"\nmagnitud > 0 = PÉRDIDA de vegetación, porque {INDICE_PRIMARIO} está invertido.")

## 7 · El mapa de degradación

Dos productos distintos, según la decisión 1.2:

- **Degradación gradual** — pérdida con magnitud > umbral y duración ≥ 3 años. El indicador principal.
- **Pérdida abrupta** — magnitud > umbral y duración ≤ 2 años. Candidatos a quema, que el notebook 07
  separará cruzando con el asset `cusco_03_quema_aparicion`.

De cada uno se extrae el segmento de mayor magnitud por píxel y sus siete atributos.

### Los valores pre y post de los índices acompañantes

Esta parte es nueva y existe para el notebook 07. Su regla de evidencia cruzada necesita saber en qué
dirección se movió **cada** índice entre el inicio y el final del segmento:

```
negTCW ↑   NDVI ↓   MSAVI ↓   BSI ↑   TCB ↑   NDMI ↓
```

Los valores ajustados de los acompañantes están en sus bandas `<nombre>_fit`, que son arrays con un
valor por año. Para leerlos en los vértices del segmento hay que indexar esos arrays por **posición**,
y la posición del año Y es `Y − 2013`. `arrayGet()` acepta una imagen de posiciones, así que la
indexación se hace por píxel: cada uno se lee en su propio año de quiebre.

Son 12 bandas (6 índices × pre/post). Sin ellas, el notebook 07 tendría que volver a correr LandTrendr
entero solo para recuperar este dato.

In [ ]:
def mapa_cambio(arr, umbral, dur_min=None, dur_max=None):
    """Bandas del segmento de pérdida principal según los filtros dados."""
    princ = principal(filtrar_perdidas(arr, umbral, dur_min, dur_max))
    return ee.Image.cat([atributo(princ, n) for n in FILAS_SEG])


def valores_pre_post(lt_img, mapa, indices=None):
    """Valor ajustado de cada índice acompañante en el vértice inicial y en el final.

    Los arrays <nombre>_fit tienen un valor por año, en orden. La posición del año Y es
    Y - ANIO_INI, y arrayGet() acepta una IMAGEN de posiciones: así cada píxel se lee en
    su propio año de quiebre.
    """
    indices = indices or INDICES_FTV
    i_ini = mapa.select("anio_ini").subtract(ANIO_INI).toInt()
    i_fin = mapa.select("anio_fin").subtract(ANIO_INI).toInt()
    bandas = []
    for nombre in indices:
        banda_fit = f"{nombre}_fit"
        if banda_fit not in bandas_lt:
            print(f"  aviso: no existe {banda_fit}, se omite")
            continue
        arr = lt_img.select(banda_fit)
        bandas.append(arr.arrayGet(i_ini).rename(f"{nombre}_pre"))
        bandas.append(arr.arrayGet(i_fin).rename(f"{nombre}_post"))
    return ee.Image.cat(bandas)


grad = mapa_cambio(segmentos, UMBRAL_MAG, dur_min=DUR_MIN)
abrupto = mapa_cambio(segmentos, UMBRAL_MAG, dur_max=DUR_QUEMA_MAX)
pre_post = valores_pre_post(lt, grad)

print("Bandas pre/post para el notebook 07:",
      con_reintentos(lambda: pre_post.bandNames().getInfo()))

# Confianza: percentil del RMSE (decisión 1.6 · se marca, no se elimina)
rmse = lt.select("rmse")
p_rmse = con_reintentos(lambda: rmse.updateMask(MASCARA_FINAL).reduceRegion(
    ee.Reducer.percentile([50, 75, 90]), aoi, ESCALA_STATS,
    maxPixels=1e10, bestEffort=True, tileScale=TILE_SCALE).getInfo())
print("\nRMSE del ajuste sobre la máscara de análisis:")
for k, v in p_rmse.items():
    print(f"  {k}: {v:.4f}")

confianza = (ee.Image(1)
             .where(rmse.gt(p_rmse["rmse_p75"]), 2)
             .where(rmse.gt(p_rmse["rmse_p90"]), 3)
             .rename("confianza"))      # 1 buena, 2 media, 3 baja

RESULTADO = (grad
             .addBands(abrupto.rename([f"ab_{n}" for n in FILAS_SEG]))
             .addBands(pre_post)
             .addBands(rmse).addBands(confianza).addBands(n_obs_buenos))

bandas_res = con_reintentos(lambda: RESULTADO.bandNames().getInfo())
print(f"\nResultado: {len(bandas_res)} bandas")
print("  7 del segmento gradual · 7 del abrupto · 12 pre/post · rmse · confianza · anios_buenos")

## 8 · Exportar el resultado a un asset

Earth Engine recalcula el grafo completo en cada petición. Exportar el resultado lo congela, y a
partir de ahí el notebook 06 (control) y el 07 (tipo de cambio) leen píxeles ya resueltos en vez de
volver a segmentar trece años.

Es lo que convierte esos dos notebooks de "al borde del límite de memoria" a "instantáneos". Una
tarea, pocos minutos.

In [ ]:
EXPORTAR_RESULTADO = False      # ponlo en True una sola vez

if EXPORTAR_RESULTADO:
    tarea = ee.batch.Export.image.toAsset(
        image=RESULTADO.toFloat().set({
            **{k: str(v) for k, v in PARAMS.items()},
            "indice_primario": INDICE_PRIMARIO,
            "indices_ftv": ",".join(INDICES_FTV),
            "umbral_magnitud": str(round(float(UMBRAL_MAG), 6)),
            "sigma_mediana": str(round(float(sigma_mediana), 6)),
            "sigmas": str(SIGMAS),
            "duracion_minima": str(DUR_MIN),
            "duracion_max_abrupta": str(DUR_QUEMA_MAX),
            "anio_ini": str(ANIO_INI), "anio_fin": str(ANIO_FIN),
            "notebook": "05_deteccion_cambio",
            "fecha_corrida": str(date.today()),
        }),
        description="cusco_05_resultado", assetId=ASSET_RESULTADO,
        region=aoi, scale=ESCALA, crs=GRID_CRS, maxPixels=1e10,
        pyramidingPolicy={".default": "mean"})
    tarea.start()
    print("Tarea lanzada:", tarea.id, "→", ASSET_RESULTADO)
    print("Seguimiento: pestaña Tasks del Code Editor, o ee.batch.Task.list()")
    print("\nCuando termine, ya se pueden correr los notebooks 06 y 07.")
else:
    print("EXPORTAR_RESULTADO = False.")
    print("Ponlo en True y ejecuta esta celda UNA VEZ para crear", ASSET_RESULTADO)
    print("Los notebooks 06 y 07 lo necesitan: sin él vuelven a segmentar desde cero.")

## 9 · Exportar a GeoTIFF

Los atributos de segmento como producto principal, más la serie ajustada del índice primario. Las
siete series ajustadas completas serían 13 × 7 valores por píxel y no compensan: los valores pre/post
en los vértices ya van en los atributos, que es lo que necesita el notebook 07.

**Este es el paquete para trabajar sin cuenta de Earth Engine**: con estos tres archivos se pueden
hacer los notebooks 07 a 10 en local.

In [ ]:
EXPORTAR = False

BANDAS_SALIDA = bandas_res


def descargar(img, ruta, dtype="float32"):
    """GeoTIFF con geemap; si falla, tarea a Google Drive."""
    try:
        geemap.download_ee_image(img, str(ruta), region=aoi, crs=GRID_CRS,
                                 scale=ESCALA, dtype=dtype, overwrite=True)
        return "descarga directa (geemap)"
    except Exception as e:
        print("  descarga directa falló:", repr(e)[:160])
        t = ee.batch.Export.image.toDrive(
            image=img, description=ruta.stem[:100], folder="cobertura_cusco",
            fileNamePrefix=ruta.stem, region=aoi, crs=GRID_CRS, scale=ESCALA, maxPixels=1e10)
        t.start()
        return "tarea a Google Drive (carpeta cobertura_cusco)"


SALIDA.mkdir(parents=True, exist_ok=True)

if EXPORTAR:
    print("1/3 · atributos de segmento")
    print("   ", descargar(RESULTADO.updateMask(MASCARA_FINAL).toFloat(),
                           SALIDA / "degradacion_segmentos.tif"))
    print("2/3 · serie ajustada del índice primario")
    print("   ", descargar(fit.updateMask(MASCARA_FINAL).toFloat(),
                           SALIDA / f"{INDICE_PRIMARIO}_ajustado.tif"))
    print("3/3 · máscara de análisis")
    print("   ", descargar(MASCARA_FINAL.unmask(0).toByte(),
                           SALIDA / "mascara_analisis.tif", "uint8"))
else:
    print("EXPORTAR = False. Ponlo en True para generar los GeoTIFF en:")
    print(" ", SALIDA)
    print("\nSe generarían 3 archivos:")
    print(f"  degradacion_segmentos.tif  · {len(BANDAS_SALIDA)} bandas")
    print(f"  {INDICE_PRIMARIO}_ajustado.tif      · {len(ANIOS)} bandas, una por año")
    print("  mascara_analisis.tif       · 1 banda, uint8")

# Los parámetros se guardan siempre: sin esto los GeoTIFF no son interpretables
registro = {**PARAMS, "indice_primario": INDICE_PRIMARIO,
            "indices_ftv": ",".join(INDICES_FTV),
            "umbral_magnitud": round(float(UMBRAL_MAG), 6),
            "sigma_mediana": round(float(sigma_mediana), 6), "sigmas": SIGMAS,
            "duracion_minima": DUR_MIN, "duracion_max_abrupta": DUR_QUEMA_MAX,
            "anio_ini": ANIO_INI, "anio_fin": ANIO_FIN,
            "asset_serie": ASSET_SERIE, "asset_mascara": ASSET_MASCARA,
            "km2_pajonal_analizable": round(km_pajonal, 2),
            "fecha_corrida": str(date.today())}
pd.Series(registro).to_csv(SALIDA / "parametros_corrida.csv", header=False)
print("\nParámetros guardados en", SALIDA / "parametros_corrida.csv")

## 10 · Resumen

In [ ]:
hay_grad = grad.select("magnitud").gt(0).unmask(0)
hay_abr = abrupto.select("magnitud").gt(0).unmask(0)

print("Superficie detectada dentro del pajonal:")
det = evaluar_serie(
    [km2_de(hay_grad.And(MASCARA_FINAL.unmask(0)).selfMask()),
     km2_de(hay_abr.And(MASCARA_FINAL.unmask(0)).selfMask())],
    ["gradual", "abrupta"])

print("=" * 78)
print("RESUMEN · notebook 05 · detección del cambio")
print("=" * 78)
print(f"Serie      : {len(ANIOS)} años ({ANIO_INI}–{ANIO_FIN}), desde el asset")
print(f"Segmenta   : {INDICE_PRIMARIO} (banda 0) · ajusta: {', '.join(INDICES_FTV)}")
print(f"Parámetros : maxSegments={PARAMS['maxSegments']}, "
      f"spikeThreshold={PARAMS['spikeThreshold']}, "
      f"recoveryThreshold={PARAMS['recoveryThreshold']}, pval={PARAMS['pvalThreshold']}")
print()
print("UMBRAL DE MAGNITUD (derivado de los datos, no elegido)")
print(f"  σ mediana del ruido en pajonal estable : {sigma_mediana:.4f}")
print(f"  Umbral = {SIGMAS}σ                             : {UMBRAL_MAG:.4f}")
print(f"  Duración mínima para degradación       : {DUR_MIN} años")
print()
print("SUPERFICIES")
print(f"  Pajonal analizable            : {km_pajonal:7.2f} km²")
print(f"  Degradación gradual detectada : {det[0]:7.2f} km²  "
      f"({det[0] / km_pajonal * 100:.1f} % del pajonal)")
print(f"  Pérdida abrupta (¿quemas?)    : {det[1]:7.2f} km²")
print()
print(f"RESULTADO: {len(bandas_res)} bandas")
print()
print("-" * 78)
print("LO QUE ESTE RESULTADO TODAVÍA NO ES")
print("-" * 78)
print("· No está validado. El RMSE es diagnóstico, no exactitud (notebook 09).")
print("· Los parámetros vienen de la literatura, no de calibración contra esta zona.")
print("· No se ha verificado si detecta de más: eso lo mide el notebook 06 comparando")
print("  la tasa de detección dentro y fuera del pajonal.")
print("· No dice QUÉ tipo de cambio es cada quiebre: degradación, quema, conversión")
print("  agrícola y expansión urbana siguen mezcladas (notebook 07).")
print("· No separa degradación antrópica de respuesta a sequía (notebook 08).")
print()
print("SIGUIENTE")
print("  1. Exportar el resultado al asset (sección 8) si no está hecho.")
print("  2. Notebook 06: controlar si podemos creerle a esta detección.")
print("=" * 78)